# Spot Check — Data Noise (Confidence < 0.70)
**Tujuan:** Periksa manual apakah label otomatis benar/salah pada data yang IndoBERT kurang yakin  

**Alur:**
1. Cell 1 — Export file spot check ke Excel
2. Anda isi kolom `label_benar` dan `label_manual` di Excel
3. Cell 2 — Hitung akurasi dan interpretasi hasil


## Cell 1 — Export Data Noise untuk Diperiksa Manual

In [ ]:
# ══════════════════════════════════════════════════════════════
# SPOT CHECK — Data Noise (confidence < 0.70)
# Tujuan: periksa manual apakah label otomatis benar/salah
# pada data yang model pelabel kurang yakin
# ══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import os

BASE      = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_method\data_labelling'
FILE_AUTO = os.path.join(BASE, 'labelled_data_final.csv')
FILE_SPOT = os.path.join(BASE, 'spot_check_noise_below07.csv')
FILE_RESULT = os.path.join(BASE, 'spot_check_noise_result.csv')

THRESHOLD   = 0.70
RANDOM_SEED = 42

# ── Load data ────────────────────────────────────────────────────────────────
df = pd.read_csv(FILE_AUTO)
print(f'Total data          : {len(df):,}')

# ── Filter data noise ─────────────────────────────────────────────────────────
df_noise = df[
    (df['confidence'] < THRESHOLD) &
    (df['source'] == 'auto')          # hanya dari label otomatis
].copy()

print(f'Data noise (< {THRESHOLD}) : {len(df_noise):,} ({len(df_noise)/len(df)*100:.2f}%)')

# ── Distribusi label pada data noise ─────────────────────────────────────────
print(f'\nDistribusi label pada data noise:')
for lbl, cnt in df_noise['label_pks'].value_counts().items():
    pct = cnt/len(df_noise)*100
    bar = chr(9608) * int(pct/3)
    print(f'  {lbl:10s}: {cnt:5,} ({pct:.1f}%) {bar}')

# ── Distribusi confidence pada data noise ─────────────────────────────────────
print(f'\nDistribusi confidence data noise:')
bins   = [0.0, 0.3, 0.4, 0.5, 0.6, 0.7]
labels_bin = ['0.0–0.3','0.3–0.4','0.4–0.5','0.5–0.6','0.6–0.7']
hist, _ = np.histogram(df_noise['confidence'], bins=bins)
for lbl, cnt in zip(labels_bin, hist):
    pct = cnt/len(df_noise)*100
    bar = chr(9608) * int(pct/5)
    print(f'  [{lbl}]: {cnt:5,} ({pct:5.1f}%) {bar}')

# ── Stratified sampling dari data noise ──────────────────────────────────────
# Ambil sampel dari SETIAP kelas agar semua terwakili
print(f'\nMenyiapkan file spot check...')

samples = []

for lbl in ['keluhan', 'saran', 'pujian']:
    df_lbl = df_noise[df_noise['label_pks'] == lbl]
    if len(df_lbl) == 0:
        continue

    # Ambil semua jika < 100, ambil proporsional jika lebih
    n_sample = min(len(df_lbl), max(30, int(len(df_lbl) * 0.1)))
    df_sample = df_lbl.sample(n=n_sample, random_state=RANDOM_SEED)
    samples.append(df_sample)
    print(f'  {lbl:10s}: {len(df_lbl):,} noise → ambil {n_sample} sampel')

df_spot = pd.concat(samples, ignore_index=True)

# Urutkan berdasarkan confidence terendah (yang paling meragukan di atas)
df_spot = df_spot.sort_values('confidence', ascending=True)

# ── Siapkan kolom untuk diisi manual ─────────────────────────────────────────
df_spot['label_manual']   = ''   # ISI INI — keluhan / saran / pujian
df_spot['label_benar']    = ''   # ISI INI — ya / tidak
df_spot['catatan']        = ''   # opsional — alasan jika berbeda

# Pilih kolom yang ditampilkan
COLS_SHOW = [
    'confidence', 'label_pks', 'text',
    'platform', 'ownerUsername',
    'label_manual', 'label_benar', 'catatan'
]
cols_available = [c for c in COLS_SHOW if c in df_spot.columns]
df_export = df_spot[cols_available].copy()

df_export.to_csv(FILE_SPOT, index=False, encoding='utf-8-sig')

print(f'\nFile spot check tersimpan: {FILE_SPOT}')
print(f'Total sampel untuk dicek  : {len(df_export):,}')
print(f'\nPanduan pengisian:')
print(f'  1. Buka file: {FILE_SPOT}')
print(f'  2. Baca kolom "text" — baca komentar aslinya')
print(f'  3. Lihat kolom "label_pks" — label otomatis dari IndoBERT')
print(f'  4. Isi kolom "label_benar":')
print(f'       → ketik "ya"   jika label_pks menurut Anda BENAR')
print(f'       → ketik "tidak" jika label_pks menurut Anda SALAH')
print(f'  5. Jika "tidak", isi "label_manual" dengan label yang benar:')
print(f'       → keluhan / saran / pujian')
print(f'  6. Simpan file lalu jalankan kode di bawah untuk hitung akurasi')

## Cell 2 — Hitung Hasil Spot Check
> Jalankan SETELAH mengisi kolom `label_benar` di file Excel

**Cara mengisi file Excel:**
| Kolom | Cara mengisi |
|---|---|
| `confidence` | Jangan diubah — nilai kepercayaan model |
| `label_pks` | Jangan diubah — label otomatis dari IndoBERT |
| `text` | Baca ini — komentar asli yang perlu dinilai |
| `label_benar` | Ketik **ya** jika label_pks benar, **tidak** jika salah |
| `label_manual` | Ketik label yang benar jika label_benar = **tidak** |
| `catatan` | Opsional — tulis alasan jika perlu |


In [ ]:
# BAGIAN 2— Hitung hasil spot check
# Jalankan SETELAH mengisi kolom label_benar di Excel
# ══════════════════════════════════════════════════════════════

def hitung_spot_check(file_result=FILE_SPOT):
    df_result = pd.read_csv(file_result)

    # Filter hanya yang sudah diisi
    df_filled = df_result[
        df_result['label_benar'].str.strip().str.lower().isin(['ya','tidak'])
    ].copy()

    if len(df_filled) == 0:
        print('Belum ada yang diisi. Isi kolom label_benar dulu di Excel.')
        return

    total      = len(df_filled)
    benar      = (df_filled['label_benar'].str.strip().str.lower() == 'ya').sum()
    salah      = total - benar
    akurasi    = benar / total * 100

    SEP = '=' * 55
    print(SEP)
    print('HASIL SPOT CHECK — DATA NOISE (confidence < 0.70)')
    print(SEP)
    print(f'  Total diperiksa  : {total}')
    print(f'  Label benar      : {benar} ({akurasi:.1f}%)')
    print(f'  Label salah      : {salah} ({100-akurasi:.1f}%)')

    print(f'\nAkurasi per kelas:')
    for lbl in ['keluhan','saran','pujian']:
        sub = df_filled[df_filled['label_pks'] == lbl]
        if len(sub) == 0:
            continue
        acc = (sub['label_benar'].str.strip().str.lower() == 'ya').mean() * 100
        print(f'  {lbl:10s}: {acc:.1f}% ({len(sub)} sampel)')

    # Analisis pola kesalahan
    df_salah = df_filled[df_filled['label_benar'].str.strip().str.lower() == 'tidak']
    if len(df_salah) > 0 and 'label_manual' in df_salah.columns:
        df_salah = df_salah[df_salah['label_manual'].str.strip() != '']
        if len(df_salah) > 0:
            print(f'\nPola kesalahan label (prediksi → seharusnya):')
            for _, row in df_salah.iterrows():
                pred   = str(row['label_pks']).strip()
                actual = str(row['label_manual']).strip()
                conf   = row['confidence']
                print(f'  [{conf:.2f}] {pred:10s} → {actual}')

    # Interpretasi
    print(f'\nInterpretasi:')
    if akurasi >= 80:
        print(f'  BAIK — {akurasi:.1f}% label noise masih benar')
        print(f'  Data noise aman dipertahankan dalam dataset')
        keputusan = 'dipertahankan'
    elif akurasi >= 60:
        print(f'  CUKUP — {akurasi:.1f}% label noise benar')
        print(f'  Pertimbangkan hapus data dengan confidence < 0.50')
        keputusan = 'pertimbangkan hapus confidence < 0.50'
    else:
        print(f'  KURANG — hanya {akurasi:.1f}% label noise benar')
        print(f'  Rekomendasi: hapus semua data confidence < 0.70')
        keputusan = 'hapus data confidence < 0.70'

    print(f'\nKalimat untuk skripsi:')
    print(f'  "Hasil spot check manual terhadap {total} sampel data')
    print(f'  dengan confidence score di bawah 0,70 menunjukkan')
    print(f'  tingkat kebenaran label sebesar {akurasi:.1f}%. Berdasarkan')
    print(f'  hasil ini, data noise {keputusan}."')

    return akurasi

# Panggil fungsi setelah file diisi
hitung_spot_check()
print('\nSetelah mengisi file Excel, uncomment baris terakhir')
print('(hapus tanda #) lalu jalankan ulang cell ini.')